# SymPyによる一般相対性理論：テンソル解析と曲率の計算

## 概要
一般相対性理論において、重力は時空の歪み（曲率）として記述される。この理論を扱うには、計量テンソルからクリストッフェル記号、リーマン曲率テンソル、リッチテンソル、そしてスカラー曲率を順次計算する必要がある。

これらの計算は、添字（インデックス）の縮約や偏微分が複雑に絡み合い、手計算で行うと膨大な労力と時間を要する。Pythonの数式処理ライブラリ **SymPy** を用いれば、これらのテンソル計算を自動化し、正確に実行することができる。

本記事では、SymPyを用いて以下のトピックを解説する。

1. **計量テンソルの定義**：シュワルツシルト計量の導入
2. **クリストッフェル記号**：接続係数の記号的導出
3. **曲率テンソル**：リーマン・リッチテンソルの計算とアインシュタイン方程式の真空解の検証

### 筆者の環境
筆者の実行環境は以下の通りである。

In [ ]:
!sw_vers

In [ ]:
!python -V

必要なライブラリをインポートする。SymPyには `sympy.tensor` モジュールがあるが、ここでは基礎的な理解のために、行列演算と微分機能を用いて明示的に実装してみる。

In [ ]:
import sympy
from sympy import symbols, sin, cos, exp, log, diff, simplify, Matrix, diag, zeros, eye
from pprint import pprint as py_pprint

# 数式をLaTeX形式で綺麗に表示するための設定
sympy.init_printing()

print("sympy version :", sympy.__version__)

## 1. 計量テンソルの定義

球対称な静的重力場を表す「シュワルツシルト計量」を考える。座標系を $x^\mu = (t, r, \theta, \phi)$ とする。
線素 $ds^2$ は以下のように与えられる。

$$ ds^2 = -\left(1 - \frac{r_s}{r}\right) dt^2 + \left(1 - \frac{r_s}{r}\right)^{-1} dr^2 + r^2 d\theta^2 + r^2 \sin^2\theta d\phi^2 $$

ここで $r_s$ はシュワルツシルト半径である。
計量テンソル $g_{\mu\nu}$ を行列として定義する。

In [ ]:
t, r, theta, phi = symbols('t r theta phi')
rs = symbols('r_s')

# 座標変数のリスト
coords = [t, r, theta, phi]

# 計量テンソルの成分
g00 = -(1 - rs/r)
g11 = 1 / (1 - rs/r)
g22 = r**2
g33 = r**2 * sin(theta)**2

# 計量テンソル g_mu_nu (共変成分)
g = diag(g00, g11, g22, g33)

print("計量テンソル g_mu_nu:")
sympy.pprint(g)

また、反変成分 $g^{\mu\nu}$ （逆行列）も必要になるため計算しておく。

In [ ]:
g_inv = g.inv()

print("計量テンソル（反変） g^mu_nu:")
sympy.pprint(g_inv)

## 2. クリストッフェル記号

第2種クリストッフェル記号 $\Gamma^\lambda_{\mu\nu}$ は計量テンソルから以下のように定義される。

$$ \Gamma^\lambda_{\mu\nu} = \frac{1}{2} g^{\lambda\sigma} (\partial_\mu g_{\nu\sigma} + \partial_\nu g_{\mu\sigma} - \partial_\sigma g_{\mu\nu}) $$

ここでアインシュタインの縮約記法を用いている（$\sigma$ について和をとる）。
これを計算する関数を作成する。

In [ ]:
def calc_christoffel(g, g_inv, coords):
    dim = len(coords)
    Gamma = [[[0 for _ in range(dim)] for _ in range(dim)] for _ in range(dim)]
    
    for lam in range(dim):
        for mu in range(dim):
            for nu in range(dim):
                sum_val = 0
                for sigma in range(dim):
                    term = diff(g[nu, sigma], coords[mu]) + \
                           diff(g[mu, sigma], coords[nu]) - \
                           diff(g[mu, nu], coords[sigma])
                    sum_val += 0.5 * g_inv[lam, sigma] * term
                Gamma[lam][mu][nu] = simplify(sum_val)
    return Gamma

Gamma = calc_christoffel(g, g_inv, coords)

# 例として Gamma^1_00 (r, t, t) 成分を表示
print("Gamma^r_tt:")
sympy.pprint(Gamma[1][0][0])

## 3. リーマン曲率テンソル

リーマン曲率テンソル $R^\rho_{\sigma\mu\nu}$ はクリストッフェル記号を用いて以下のように定義される。

$$ R^\rho_{\sigma\mu\nu} = \partial_\mu \Gamma^\rho_{\nu\sigma} - \partial_\nu \Gamma^\rho_{\mu\sigma} + \Gamma^\rho_{\mu\lambda} \Gamma^\lambda_{\nu\sigma} - \Gamma^\rho_{\nu\lambda} \Gamma^\lambda_{\mu\sigma} $$

これも定義通りに計算する。

In [ ]:
def calc_riemann(Gamma, coords):
    dim = len(coords)
    Riemann = [[[[0 for _ in range(dim)] for _ in range(dim)] for _ in range(dim)] for _ in range(dim)]
    
    for rho in range(dim):
        for sigma in range(dim):
            for mu in range(dim):
                for nu in range(dim):
                    term1 = diff(Gamma[rho][nu][sigma], coords[mu])
                    term2 = diff(Gamma[rho][mu][sigma], coords[nu])
                    
                    term3 = 0
                    term4 = 0
                    for lam in range(dim):
                        term3 += Gamma[rho][mu][lam] * Gamma[lam][nu][sigma]
                        term4 += Gamma[rho][nu][lam] * Gamma[lam][mu][sigma]
                    
                    Riemann[rho][sigma][mu][nu] = simplify(term1 - term2 + term3 - term4)
    return Riemann

Riemann = calc_riemann(Gamma, coords)

# 非ゼロ成分の例を表示
print("R^t_rtr:")
sympy.pprint(Riemann[0][1][0][1])

## 4. リッチテンソルとアインシュタイン方程式

リッチテンソル $R_{\mu\nu}$ はリーマンテンソルの縮約である。

$$ R_{\mu\nu} = R^\lambda_{\mu\lambda\nu} $$

シュワルツシルト解は真空解であるため、アインシュタイン方程式 $R_{\mu\nu} = 0$ を満たすはずである。これを検証する。

In [ ]:
def calc_ricci(Riemann, coords):
    dim = len(coords)
    Ricci = [[0 for _ in range(dim)] for _ in range(dim)]
    
    for mu in range(dim):
        for nu in range(dim):
            sum_val = 0
            for lam in range(dim):
                sum_val += Riemann[lam][mu][lam][nu]
            Ricci[mu][nu] = simplify(sum_val)
    return Ricci

Ricci = calc_ricci(Riemann, coords)

print("リッチテンソル R_mu_nu:")
py_pprint(Ricci)

すべての成分が 0 となった。これにより、シュワルツシルト計量が真空中のアインシュタイン方程式の厳密解であることが、SymPyを用いた記号計算によって証明された。

## 結論

この記事では、SymPyを用いて一般相対性理論におけるテンソル解析を行った。

計量テンソルの定義から出発し、クリストッフェル記号、リーマン曲率テンソル、そしてリッチテンソルを順次計算することで、複雑なテンソル計算を自動化できることを示した。特に、手計算では数ページに及ぶ計算が、わずか数十行のコードで完結し、かつ計算ミスのリスクがない点は、理論物理学の研究において極めて強力な利点となる。

### 参考文献
- [SymPy Documentation: Tensor Module](https://docs.sympy.org/latest/modules/tensor/index.html) (今回は理解のために自作関数を用いたが、専用モジュールも存在する)